# Demo: ledger2 seal

How does a value made by hand become something the ledger vouches
for? `t.seal(v, deps=[...])` freezes a `Value` into an immutable
subject named by its content and its declared inputs, and writes
rows saying what it is and what it was made from. This notebook
classifies the shots of game G5 by hand, seals the table against
the run and the model it used, gives it a readable name, and loads
it back.

In [1]:
import inspect, os, shutil, sys, tempfile
from pathlib import Path

import pandas as pd

sys.path.insert(0, "demo_svp")          # the stand-in svp package
from dvc_dat.ledger2 import Code, Env, Exists, Ledger
from dvc_dat.ledger2 import set_env_provider
from svp.shots import Shots

ENV = Env({"sv-pipe": Code("0.2.2", "7d742989")},
          lock="sha256:5e1f", platform="darwin-arm64-py3.11")
old_provider = set_env_provider(lambda module: ENV)
ledger_dir = Path(tempfile.mkdtemp(prefix="demo-l2-seal-"))
os.environ["DAT_LEDGER"] = str(ledger_dir)
L = Ledger.open()

The environment is pinned (sv-pipe 0.2.2 at `7d742989`) so the
output is the same on every machine.

## A hand-made `Value`

In [2]:
print(inspect.getsource(Shots))

class Shots(Value):
    def __init__(self, table: pd.DataFrame):
        self.table = table

    def encode(self) -> bytes:
        return self.table.to_parquet()

    @classmethod
    def decode(cls, data, store):
        return cls(pd.read_parquet(io.BytesIO(data)))



A `Value` subclass says how it becomes bytes and back. It lives in
a module, `svp.shots`, because a loaded value finds its class by
dotted name.

In [3]:
frames = pd.DataFrame({"frame": [120, 388, 902],
                       "x": [0.41, 0.77, 0.52]})
with L.transaction(by="human:dan") as t:
    t.add("code/sv-pipe", "version", "0.2.2")
    t.add("code/sv-pipe", "commit", "7d742989")
    t.add("model/shot", "version", "1.0.0")
    t.add("model/shot", "value", {"threshold": 0.6})
    t.add("run/0-2-2/G5", "value", Shots(frames))
rev = L.at()
run, model = rev.load("run/0-2-2/G5"), rev.load("model/shot")
print(type(run).__name__, run.subject, run.sealed)
print(type(model).__name__, model.obj, model.sealed)

Shots run/0-2-2/G5 False
Pickled {'threshold': 0.6} False


`add(s, "value", v)` stores the bytes by hash and points the
subject at them; `rev.load(s)` brings the object back. Neither is
sealed: a `value` row can be replaced by the next `add`.

## `seal`

In [4]:
def classify(run, model):
    made = run.table["x"] > model.obj["threshold"]
    return run.table.assign(made=made)


hand = Shots(classify(run, model))
with L.transaction(by="human:dan", time=True) as t:
    subj = t.seal(hand, deps=[run, model])
print(subj, hand.sealed, hand.subject == subj)
for item, value in hand.trace.items():
    print(f"  {item:22} {value[:30]}")

Shots/1695f598acfe28a3 True True
  dat:run/0-2-2/G5       sha256:0c5e2fcb19ace10cd56b165
  dat:model/shot         sha256:77717a8d9b5a21b12feac55
  code/sv-pipe           0.2.2+7d742989
  uv.lock                sha256:5e1f
  platform               darwin-arm64-py3.11


The subject is `<kind>/<hash>`, hashed over the bytes and the trace.
The trace is what the value was made from: each declared dep by its
hash, plus the code, the lock and the platform it was made in.

In [5]:
for p, o in L.at().all(subj):
    print(f"  {p:24} {str(o)[:40]}")

  value                    Ref(2, 'Shots/1695f598acfe28a3')
  kind                     Shots
  provenance               seal
  trace.code/sv-pipe       0.2.2+7d742989
  trace.dat:model/shot     sha256:77717a8d9b5a21b12feac55c9895cc502
  trace.dat:run/0-2-2/G5   sha256:0c5e2fcb19ace10cd56b165be4fea7bb2
  trace.platform           darwin-arm64-py3.11
  trace.uv.lock            sha256:5e1f


The rows `seal` wrote: the `value` (an artifact `Ref`), the `kind`,
the `provenance` (`seal`, the verb), and one `trace.` row per item.

## Sealed is immutable

In [6]:
def add_row(p, o):
    with L.transaction(by="human:dan") as t:
        t.add(subj, p, o)


try:
    add_row("kind", "Tracks")
except Exists as e:
    print("Exists:", e)
add_row("note", "by hand, G5")
print(L.at()[subj, "note"])

Exists: 'Shots/1695f598acfe28a3' is sealed: 'kind' is one of its own predicates, written once (annotate it with any other predicate)
by hand, G5


A sealed subject's own predicates are written once: rewriting one
raises `Exists` at commit. Any other predicate is an annotation and
is free.

In [7]:
rev = L.at()
run, model = rev.load("run/0-2-2/G5"), rev.load("model/shot")
again = Shots(classify(run, model))
with L.transaction(by="human:dan") as t:
    same = t.seal(again, deps=[run, model])
print(same == subj, t.rev.all(since=L.at(t.rev.id - 1)))

True [('ledger/tx/4', 'by', 'human:dan')]


Sealing the same bytes with the same deps is the same subject, and
writes nothing but the transaction's `by` row. (A dep loaded at an
older revision than `t.base` is traced as `dat:<subject>@<rev>`,
which makes a different subject; so the deps are reloaded here.)

## A readable name is a pointer

In [8]:
with L.transaction(by="human:dan") as t:
    t.add("shots/G5-byhand", "value", hand)
rev = L.at()
print(rev["shots/G5-byhand", "value"])
back = rev.load("shots/G5-byhand")
print(back.subject, back.sealed, back.same_as(hand))
print(back.table.to_string(index=False))

Ref(5, 'Shots/1695f598acfe28a3')
Shots/1695f598acfe28a3 True True
 frame    x  made
   120 0.41 False
   388 0.77  True
   902 0.52 False


`shots/G5-byhand` holds a `value` row that points at the sealed
subject. `rev.load(name)` follows it and hands back the sealed value
with its subject and trace restored.

In [9]:
with L.transaction(by="human:dan") as t:
    t.add("model/shot", "value", {"threshold": 0.5})
print("still holds at rev:", back.matches(back.rev))
print("holds at the top:  ", back.matches(L.at()))

still holds at rev: True
holds at the top:   False


`matches(rev)` re-checks every trace item at a revision: the model
it declared has moved since, so at the top it no longer holds. The
sealed subject itself is unchanged.

In [10]:
set_env_provider(old_provider)
shutil.rmtree(ledger_dir)
del os.environ["DAT_LEDGER"]
print("removed:", not ledger_dir.exists())

removed: True
